# UAVBench Getting Started

This notebook demonstrates how to:
1. Load a scenario and run A* planning
2. Visualize the planned path
3. Compare difficulty levels
4. Compute operational metrics

In [ ]:
from uavbench.cli.benchmark import run_planner_once, aggregate
from uavbench.metrics.operational import compute_all_metrics
import numpy as np
import matplotlib.pyplot as plt

## 1. Run A* on a Single Scenario

In [ ]:
result = run_planner_once("osm_athens_wildfire_easy", "astar", seed=42)

print(f"Success: {result['success']}")
print(f"Path length: {result['path_length']}")
print(f"Planning time: {result['planning_time']*1000:.1f}ms")
print(f"Start: {result['start']}, Goal: {result['goal']}")

## 2. Visualize the Path

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

# Show heightmap
ax.imshow(result['heightmap'], cmap='terrain', origin='upper')

# Overlay no-fly zones
nfz_overlay = np.zeros((*result['no_fly'].shape, 4))
nfz_overlay[result['no_fly']] = [1, 0, 0, 0.3]
ax.imshow(nfz_overlay, origin='upper')

# Draw path
if result['path']:
    xs = [p[0] for p in result['path']]
    ys = [p[1] for p in result['path']]
    ax.plot(xs, ys, 'cyan', linewidth=1.5, alpha=0.8)

# Mark start and goal
ax.plot(*result['start'], 'go', markersize=10, label='Start')
ax.plot(*result['goal'], 'r*', markersize=12, label='Goal')

ax.set_title(f"A* on {result['scenario']} (len={result['path_length']})")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Compare Difficulty Levels

In [ ]:
difficulties = ["easy", "medium", "hard"]
results = {}

for diff in difficulties:
    scenario_id = f"osm_athens_wildfire_{diff}"
    r = run_planner_once(scenario_id, "astar", seed=42)
    results[diff] = r
    print(f"{diff:>8}: success={r['success']}, path_len={r['path_length']}, "
          f"plan_time={r['planning_time']*1000:.1f}ms")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, diff in zip(axes, difficulties):
    r = results[diff]
    ax.imshow(r['heightmap'], cmap='terrain', origin='upper')
    if r['path']:
        xs = [p[0] for p in r['path']]
        ys = [p[1] for p in r['path']]
        ax.plot(xs, ys, 'cyan', linewidth=1.5)
    ax.plot(*r['start'], 'go', markersize=8)
    ax.plot(*r['goal'], 'r*', markersize=10)
    ax.set_title(f"{diff.upper()} (len={r['path_length']})")

plt.suptitle("Wildfire Scenario: Difficulty Comparison", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Compute Operational Metrics

In [ ]:
print(f"{'Metric':<30} {'Easy':>8} {'Medium':>8} {'Hard':>8}")
print("-" * 56)

all_metrics = {diff: compute_all_metrics(results[diff]) for diff in difficulties}

for key in sorted(all_metrics['easy'].keys()):
    vals = [all_metrics[d][key] for d in difficulties]
    print(f"{key:<30} {vals[0]:>8.4f} {vals[1]:>8.4f} {vals[2]:>8.4f}")

## 5. Aggregate Across Trials

In [ ]:
# Run 5 trials of the hard scenario
trials = []
for seed in range(5):
    r = run_planner_once("osm_athens_wildfire_hard", "astar", seed=seed)
    trials.append(r)
    print(f"  seed={seed}: success={r['success']}, len={r['path_length']}")

agg = aggregate(trials)
print(f"\nAggregated over {len(trials)} trials:")
print(f"  Success rate: {agg['success_rate']:.0%}")
print(f"  Avg path length: {agg['avg_path_length']:.0f}")
print(f"  Avg optimality: {agg.get('avg_path_optimality', 0):.3f}")
print(f"  Avg smoothness: {agg.get('avg_path_smoothness', 0):.3f}")
print(f"  Avg planning time: {agg.get('avg_planning_time_ms', 0):.1f}ms")